In [1]:
%pip install -q langchain langchain-community langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q ollama
%pip install -q chromadb
%pip install -q langchain

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -q langchain-openai langchain_chroma

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -q unstructured

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -q rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install -q JPype1

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install -q konlpy

Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install -qU langchain-ollama
%pip install -U ollama

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [37]:
from dotenv import load_dotenv
import os
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGSMITH_ENDPOINT= os.getenv('LANGSMITH_ENDPOINT')
LANGSMITH_PROJECT= os.getenv('LANGSMITH_PROJECT')
LANGSMITH_TRACING= os.getenv('LANGSMITH_TRACING')
LANGSMITH_API_KEY= os.getenv('LANGSMITH_API_KEY')

if LANGSMITH_ENDPOINT is None:
    raise ValueError("LANGSMITH_ENDPOINT is not set. Please set it in yont variables.")

ValueError: LANGSMITH_ENDPOINT is not set. Please set it in yont variables.

# 문서 로드, 분할

In [39]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from konlpy.tag import Okt

In [40]:
okt = Okt()

def len_okt(text):
    tokens = [token for token in okt.morphs(text)]

    return len(tokens)

def okt_tokenize(text):
    return [token for token in okt.morphs(text)]

# 텍스트 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

input_chunk_size = 1000
input_chunk_overlap = 100
text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " ", ""],
    chunk_size = input_chunk_size,
    chunk_overlap = input_chunk_overlap,
    length_function = len_okt
)

# 경로 지정, 문서 로드
direct = "../../Data_Files"
filetype = 'txt'
print(f"디렉토리 경로'{direct}' 아래 '{filetype}'형식의 파일들을 로드합니다.\n")
loader = DirectoryLoader(direct ,glob="*."+filetype, loader_cls=TextLoader)
docs = loader.load()
print("로드 된 파일 개수 :", len(docs))

# 문서 분할
print(f"청크사이즈{input_chunk_size}, 청크오버랩 사이즈{input_chunk_overlap} 인 text_splitter를 실행합니다.")
texts = text_splitter.split_documents(docs)
print("문서 분할 완료")
print("총 청크 개수: ",len(texts))


디렉토리 경로'../../Data_Files' 아래 'txt'형식의 파일들을 로드합니다.

로드 된 파일 개수 : 2
청크사이즈1000, 청크오버랩 사이즈100 인 text_splitter를 실행합니다.
문서 분할 완료
총 청크 개수:  7594


# 임베딩 또는 기존 벡터db사용
- 기존 벡터 db사용 ?가능한가 확인해 봐야함

In [41]:
# 임베딩
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = 'nlpai-lab/KURE-v1'
model_name1 = 'KURE-v1' # 폴더 디렉토리에 들어갈 변수, /사용 못함
# model_name = 'jhgan/ko-sbert-nli'
model_kwargs = {'device': 'cpu'} # cpu 사용하려면
# model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}
print(f"hf_embedding_model설정, 모델 이름 : {model_name}\n")
hf_embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

hf_embedding_model설정, 모델 이름 : nlpai-lab/KURE-v1



In [42]:
# 임베딩 벡터 경로 지정
# 파일 형식(txt/json)_청크-오버랩_임베딩모델
embedding_directory = '../../Data_Files/Vectorstore'
collectionName = filetype+'_'+str(input_chunk_size)+'-'+str(input_chunk_overlap)+'_'+model_name1
print(f"임베딩 벡터db 경로: {embedding_directory}")
print(f"벡터db collectionName: {collectionName}")


임베딩 벡터db 경로: ../../Data_Files/Vectorstore
벡터db collectionName: txt_1000-100_KURE-v1


In [ ]:
import os
from langchain_chroma import Chroma
# save_directory 가 존재한다면 해당 경로를 db로 설정
if os.path.exists(str(embedding_directory+'/'+collectionName)):
    # 기존 DB 로드
    db = Chroma(collection_name=collectionName,
                embedding_function=hf_embedding_model,
                persist_directory=embedding_directory)
    print("기존 Chroma DB를 로드했습니다.")
else:
    # 새 DB 생성 및 저장
    db = Chroma.from_documents(texts, 
                               hf_embedding_model, 
                               collection_name=collectionName, 
                               persist_directory=embedding_directory)
    print("새로운 Chroma DB를 생성하고 저장했습니다.")

# 검색 retriever

In [ ]:
# 문서 검색기 1
k_num = 3
retriever = db.as_retriever(
    search_kwargs = {'k': k_num}
)

# 문서 검색기 2
from langchain_community.retrievers import BM25Retriever
bm_k_num = 1
bm_retriever = BM25Retriever.from_documents(
    documents = docs,
    preprocess_func = okt_tokenize,
)

bm_retriever.k = bm_k_num

# 앙상블 retriever
from langchain.retrievers import EnsembleRetriever
retriever1_rate = 0.5
ensemble_retriever = EnsembleRetriever(
    retrievers = [retriever, bm_retriever],
    weights = [retriever1_rate, 1-retriever1_rate]
)

# 검색 문서 포맷
def format_docs(docs):
    return "\n\n".join(document.page_content for document in docs)

# 프롬프트, LLM 모델 지정

In [ ]:
# 프롬프트
from langchain_core.prompts import ChatPromptTemplate

prompt_content = """
            You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If you don't know the answer, just say that you don't know.

            Answer in Korean.

            #Context:
            {context}
            """

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            prompt_content,
        ), ("human", "{question}"),
    ]
)

In [ ]:
# from langchain_openai import ChatOpenAI
# from langchain_community.llms import OllamaLLM
from langchain_ollama import ChatOllama
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

LLM_model="gemma3:4b"
llm = ChatOllama(
    model = LLM_model,
    temperature = 0,
    streaming = True,
    callback_manager = CallbackManager([StreamingStdOutCallbackHandler()],)
)

# LLM 체인

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

chain = {
    "context" : ensemble_retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough()
} | prompt | llm | StrOutputParser()

In [ ]:
question = '프론트 엔드를 뽑는 채용공고를 알려줘.'
response = chain.invoke(question)

## 프론트엔드 개발자 채용 공고 (예시)

**[회사명]**은 혁신적인 서비스를 통해 사용자 경험을 개선하고, 더 나은 세상을 만드는 데 기여하는 것을 목표로 합니다. 사용자 중심의 디자인과 기술을 바탕으로 빠르게 성장하고 있으며, 함께 성장하며 미래를 만들어갈 프론트엔드 개발자를 찾고 있습니다.

**1. 채용 분야:** 프론트엔드 개발자

**2. 주요 업무:**

*   웹/앱 서비스의 프론트엔드 개발 및 유지보수
*   UI/UX 디자이너와 협업하여 사용자 인터페이스 구현
*   백엔드 API 연동 및 데이터 처리
*   웹 표준 및 최신 기술 트렌드 적용
*   코드 품질 향상을 위한 코드 리뷰 및 테스트 수행
*   새로운 기능 개발 및 개선

**3. 자격 요건:**

*   프론트엔드 개발 경력 1년 이상 (신입 가능)
*   HTML, CSS, JavaScript에 대한 깊이 있는 이해
*   React, Vue.js, Angular 등 현대적인 프레임워크 사용 경험 (필수)
*   RESTful API 연동 경험
*   웹 표준 및 웹 접근성에 대한 이해
*   Git 등 버전 관리 시스템 사용 경험
*   문제 해결 능력 및 꼼꼼한 업무 처리 능력
*   협업 능력 및 커뮤니케이션 능력

**4. 우대 조건:**

*   백엔드 개발 경험
*   테스트 자동화 경험
*   웹 성능 최적화 경험
*   UI/UX 디자인 관련 경험
*   오픈 소스 프로젝트 참여 경험
*   영어 자기소개 가능자

**5. 근무 조건:**

*   고용 형태: 정규직 (수습 기간 3개월)
*   급여: 회사 내규에 따름 (경력 및 능력에 따라 협의)
*   근무 시간: 주 5일, 9시 ~ 6시 (유연 근무 가능)
*   근무 장소: [회사 주소]
*   복리후생: 4대 보험, 퇴직 연금, 건강 보험, 교육 지원, 자기 개발비 지원, 식사 제공, 휴가 제도 등

**6. 지원 방법:**

*   제출 서류: 이력서, 자기소개서 (경력 및 기술 스택 상세 기재)

# 평가